# Backbone explorer — build any backbone × dataset × subset

Parameterised rewrite of `yolo_backbone_swap`: builds the detector for the chosen
`BACKBONE` via the unified `train_detector` driver, inspects its structure / feature
dims, and runs a short train+val on `(FAMILY, SUBSET)`. Same parameter block as the
playground and crop-training notebooks — change `BACKBONE`/`FAMILY`/`SUBSET` to switch.

## Parameters

In [2]:
# ==================== PARAMETERS (identical block in every 02_2stage notebook) ====================
# Switch the whole notebook by editing these three. BACKBONE is any key from
# train_detector.BACKBONES; FAMILY/SUBSET pick the dataset subset.
BACKBONE = "dinov2"      # resnet34 resnet50 convnext convnext_dinov3
                          # vit_small_patch16_224.augreg_in21k vit_small_patch16_dinov3
                          # dinov2 dinov2_reg am-radio
FAMILY   = "diva"        # "diva" | "udiads"
SUBSET   = "CB55"        # diva: CB55 CS18 CS863   |   udiads: Latin14396 Latin2 Syr341
IMGSZ    = 1280           # detector inference/training size (multiple of 32)
CONF     = 0.25           # detector confidence threshold
CROP_IMGSZ = 640          # stage-2 crop-segmenter size

## Resolve paths

In [3]:
import sys, torch
from pathlib import Path
sys.path.insert(0, ".")
import train_detector

assert BACKBONE in train_detector.BACKBONES, f"unknown BACKBONE {BACKBONE!r}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# (FAMILY, SUBSET) -> detection data config + trained-detector checkpoint path.
DATA_CFG = Path(f"configs/data/{('diva_'+SUBSET.lower()) if FAMILY=='diva' else ('udiads_'+SUBSET.lower())}_detect.yaml")
FAM_DIR = "diva-hisdb" if FAMILY == "diva" else "u-diads-tl"
RUN = f"yolov8_{BACKBONE}_{FAMILY}_{SUBSET}"
DET_WEIGHTS = Path(f"../../80_models/instance_segmentation/rtdetr_bbox_unet/{FAM_DIR}/detection/{RUN}/weights/best.pt")
print("device      :", DEVICE)
print("data config :", DATA_CFG, "(exists)" if DATA_CFG.exists() else "(MISSING)")
print("detector    :", DET_WEIGHTS, "(exists)" if DET_WEIGHTS.exists() else "(train it first)")
# Absolute data YAML (fixes Ultralytics resolving `path:` vs datasets_dir).
DATA = train_detector.resolve_data(DATA_CFG) if DATA_CFG.exists() else str(DATA_CFG)


ModuleNotFoundError: No module named 'hier_encoder'

## Build the detector for BACKBONE and inspect it

In [ ]:
model = train_detector.build_model(BACKBONE, nc=1, freeze=True, scale="s")
net = model.model
print(net)  # Ultralytics layer table

# Detection pyramid: strides [8,16,32] => 3 levels (P3/P4/P5) for every backbone.
# (In eval mode the full net returns a (preds, feats) tuple, so probe strides instead.)
print("\nDetection strides:", [int(s) for s in net.stride], "-> ", len(net.stride), "pyramid levels")

## Quick train + val on the chosen subset

In [ ]:
# Trains + saves under 80_models/instance_segmentation/rtdetr_bbox_unet/<family>/detection/<RUN> (same place
# run_train_all.sh writes, and where cell-10 val looks). Mirrors that recipe.
model.train(
    data=DATA, epochs=100, imgsz=IMGSZ,
    project=f"../../80_models/instance_segmentation/rtdetr_bbox_unet/{FAM_DIR}/detection", name=RUN, exist_ok=True,
)

NameError: name 'model' is not defined

## Validate a trained checkpoint

In [ ]:
if DET_WEIGHTS.exists():
    from ultralytics import YOLO
    m = YOLO(str(DET_WEIGHTS))
    metrics = m.val(data=DATA, imgsz=IMGSZ)
    print("mAP50:", metrics.box.map50, " mAP50-95:", metrics.box.map)
else:
    print("No trained checkpoint yet at", DET_WEIGHTS)